In [1]:
from dummy_app.models.simulation_builder import Builder 
from dummy_app.designs.envsim import EnvSim
from dummy_app.designs.mobility import GroundUserGroup
from dummy_app.designs.voronoi_map import Map 
from dummy_app.tools.logger import logger 

import os 
import numpy as np 
from pathlib import Path 


PROJECT_DIR = os.getcwd() 
PROJECT_ASSETS = f"{PROJECT_DIR}/assets"

max_memory = 2 * 1024 *1024 *1024
number_of_agents = 5
max_battery = 1500
constraints = [] # Here have the constraints to form the optimization problem. 

config = {
    "regionalization":True, 
    "genetic_algorithm": True, 
    "individual_solution": False, 
    "constraints":constraints,
}

dist_path = f"{PROJECT_ASSETS}/env_settings/distance_cost.csv"
energy_path = f"{PROJECT_ASSETS}/env_settings/energy_cost.csv"
areas_path = f"{PROJECT_ASSETS}/env_settings/areas.csv"
customers_path = f"{PROJECT_ASSETS}/env_settings/customers.csv"
gues_path = f"{PROJECT_ASSETS}/env_settings/ground_users.csv"


In [2]:
#Set up the problem builder (the main builder for the program) 
builder = Builder(config)

#Set up simulation environment with simpy 
mobility_sim = EnvSim() 

#Set up the map for the simulation 
map = Map(
    data_path=areas_path, 
    incremental=False
)
map.voronoi_tessellation() 

#Set up ground users as groups within the same areas. 
groun_users = GroundUserGroup(
    env=mobility_sim.env, 
    map_obj=map, 
    alpha=0.85, 
    mean_velocity=1.0, 
    sigma=0.5
)

# Load Users from the csv. These are created based on matlab simulation. 
groun_users.load_users(
    data_path=gues_path, 
    customers_path=customers_path
)


In [3]:
# Prepare the data for the problem solution: 

data = builder.preprocess(
    distances_path=dist_path, 
    energies=energy_path,
    nodes_path=areas_path,
    agents=number_of_agents,
    customers_path=customers_path, 
    ground_users=gues_path, 
    max_battery=max_battery 
)


2025-05-12 12:59:39 | INFO | lstm_forecaster | Preprocessing completed successfully...


In [ ]:
trials = 3600 
mobility_sim.simulations(
    constructor=builder,
    cues=groun_users,
    data=data,
    trials=trials
) 



AttributeError: 'Environment' object has no attribute 'simulations'